# Lesson 3 — BGE-base dense retrieval on MS MARCO (Anserini)
Official doc: https://github.com/castorini/anserini/blob/master/docs/experiments-msmarco-passage2.md
Pinned to Anserini commit `92f8cb98ec54ce891f507dc01b22247eae34127b` (same commit as Lessons 1–2).

**How to run:** Runtime → Run all. Takes roughly 40–80 minutes (most of it is the 26 GB download). Expected result: **MRR@10 = 0.3521**.
At the end, the last cell prints a summary. Copy it back to Claude.

In [ ]:
# 1. Check the machine (record this; it goes in the PR)
!cat /etc/os-release | head -2; nproc; free -g | grep Mem; df -h / | tail -1
!nvidia-smi -L 2>/dev/null || echo "no GPU (fine, not needed)"
import shutil
assert shutil.disk_usage('/').free > 60e9, "Need ~60 GB free disk for the 26 GB index (tarball + extracted)."


In [ ]:
# 2. Java 21 + Maven 3.9 (Anserini requires Java 21)
!apt-get -qq update > /dev/null && apt-get -qq install -y openjdk-21-jdk-headless > /dev/null
!cd /opt && wget -q https://archive.apache.org/dist/maven/maven-3/3.9.11/binaries/apache-maven-3.9.11-bin.tar.gz && tar xzf apache-maven-3.9.11-bin.tar.gz
import os
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'
os.environ['PATH'] = '/opt/apache-maven-3.9.11/bin:' + os.environ['JAVA_HOME'] + '/bin:' + os.environ['PATH']
os.environ['LANG'] = 'C.UTF-8'
!java -version 2>&1 | head -1; mvn -v | head -1


In [ ]:
# 3. Anserini at the pinned commit, built without re-running the test suite (already run in Lesson 2)
%cd /content
!git clone -q https://github.com/castorini/anserini.git
%cd /content/anserini
!git checkout -q 92f8cb98ec54ce891f507dc01b22247eae34127b && git log -1 --format='%H %cd'
!mvn -q clean package -DskipTests -Dslf4j.internal.verbosity=WARN && ls target/*fatjar.jar


In [ ]:
# 4. Queries + qrels (same files as Lesson 1, MD5-checked)
!mkdir -p collections/msmarco-passage runs
!wget -q https://msmarco.z22.web.core.windows.net/msmarcoranking/collectionandqueries.tar.gz -P collections/msmarco-passage
!md5sum collections/msmarco-passage/collectionandqueries.tar.gz   # expect 31644046b18952c1386cd4564ba2ae69
!tar xzf collections/msmarco-passage/collectionandqueries.tar.gz -C collections/msmarco-passage
!rm collections/msmarco-passage/collectionandqueries.tar.gz collections/msmarco-passage/collection.tsv
!python src/main/python/msmarco/convert_msmarco_to_trec_qrels.py --input collections/msmarco-passage/qrels.dev.small.tsv --output collections/msmarco-passage/qrels.dev.small.trec
!wc -l collections/msmarco-passage/queries.dev.small.tsv   # expect 6980


## 5. The experiment
Same class and flags as the doc's `bin/run.sh io.anserini.search.SearchHnswDenseVectors ...` command. The only change: `java` is called directly with a heap that fits Colab's RAM, because `bin/run.sh` asks for `-Xmx192G`. The HNSW vectors are memory-mapped outside the heap, so the heap size doesn't affect results.
First run downloads the 26 GB prebuilt index into `~/.cache/pyserini/indexes`.

In [ ]:
# 5–7. Search + evaluation + summary in ONE cell, so the result prints as soon as the search ends
import psutil, subprocess, datetime
heap = max(4, int(psutil.virtual_memory().total / 1e9 * 0.4))
print('JVM heap:', heap, 'GB', flush=True)
!time java -cp target/anserini-2.3.1-SNAPSHOT-fatjar.jar -Xms512M -Xmx{heap}G --add-modules jdk.incubator.vector --enable-native-access=ALL-UNNAMED io.anserini.search.SearchHnswDenseVectors -index msmarco-v1-passage.bge-base-en-v1.5.hnsw -topics collections/msmarco-passage/queries.dev.small.tsv -topicReader TsvInt -output runs/run.msmarco-passage.dev.bge.txt -encoder BgeBaseEn15 -hits 1000 -threads 4 > runs/search.log 2>&1
!tail -15 runs/search.log
sh = lambda c: subprocess.run(c, shell=True, capture_output=True, text=True).stdout.strip()
print('===== SUMMARY (copy this to Claude) =====')
print('date:', datetime.date.today())
print('commit:', sh('git rev-parse HEAD'))
print('os:', sh("grep PRETTY_NAME /etc/os-release"))
print('cpus:', sh('nproc'), '| ram:', sh("free -g | awk '/Mem/{print $2}'"), 'GB | gpu:', sh('nvidia-smi -L 2>/dev/null') or 'none')
print('java:', sh('java -version 2>&1 | head -1'))
print('run lines:', sh('wc -l < runs/run.msmarco-passage.dev.bge.txt'))
print(sh('bin/trec_eval -c -M 10 -m recip_rank collections/msmarco-passage/qrels.dev.small.trec runs/run.msmarco-passage.dev.bge.txt'))
print('===== DONE =====')
